# XClinVision: Data Preprocessing Pipeline
**Day 1: EDA → Data Preparation**

This notebook runs the processing pipeline from `xclinvision.processing`:
1. Scans `data/raw/` for all images
2. Runs quality‑control checks (corrupted, blank, borders, blur, size)
3. Quarantines bad images → `data/quarantine/`
4. Preprocesses clean images → `data/processed/`
5. Validates the output

## 1. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from xclinvision.processing import (
    load_dataset_metadata,
    run_processing_pipeline,
    PipelineReport,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
QUARANTINE_DIR = Path("../data/quarantine")
DUPLICATE_DIR = Path("../data/duplicate")

print(f"Raw data:       {RAW_DIR}  (exists: {RAW_DIR.exists()})")
print(f"Processed dir:  {PROCESSED_DIR}")
print(f"Quarantine dir: {QUARANTINE_DIR}")
print(f"Duplicate dir:  {DUPLICATE_DIR}")

Raw data:       ../data/raw  (exists: True)
Processed dir:  ../data/processed
Quarantine dir: ../data/quarantine


## 2. Run the Processing Pipeline
This scans every image in `data/raw/`, runs quality-control checks (cropping, aspect ratio, blank detection), preprocesses the clean ones (CLAHE + resize to 224×224), and quarantines the rest.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

report = run_processing_pipeline(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    quarantine_dir=QUARANTINE_DIR,
    duplicate_dir=DUPLICATE_DIR,
    target_size=224,
)

INFO | Scanning raw dataset at ../data/raw …
INFO | Found 25553 images across 3 splits
INFO | Running quality‑control checks on 25553 images …


QC checks:   0%|          | 0/25553 [00:00<?, ?it/s]

INFO | QC summary — clean: 11460 | flagged: 14092 | corrupted: 1


Quarantining corrupted:   0%|          | 0/1 [00:00<?, ?it/s]

Quarantining flagged:   0%|          | 0/14092 [00:00<?, ?it/s]

INFO | Quarantine manifest saved to ../data/quarantine/quarantine_manifest.csv


Processing clean images:   0%|          | 0/11460 [00:00<?, ?it/s]


PROCESSING PIPELINE COMPLETE
  Total raw images:          25553
  Processed (clean):         11460
  Quarantined (corrupted):   1
  Quarantined (flagged):     14092

  Processed dir:   ../data/processed
  Quarantine dir:  ../data/quarantine
  Manifest:        ../data/processed/manifest.csv
  Quarantine log:  ../data/quarantine/quarantine_manifest.csv


## 3. Inspect the Processed Manifest
Load the manifest CSV and verify the class/split distribution matches expectations.

In [ ]:
manifest = pd.read_csv(PROCESSED_DIR / "manifest.csv")
print(f"Processed images: {len(manifest)}\n")

# Split × class breakdown
pivot = manifest.groupby(["split", "class"]).size().unstack(fill_value=0)
print(pivot)
print()

# Quick bar chart
pivot_melted = pivot.reset_index().melt(id_vars="split", var_name="class", value_name="count")
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=pivot_melted, x="split", y="count", hue="class", ax=ax)
ax.set_title("Processed Dataset — Class Counts per Split")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Inspect the Quarantine Manifest
See which images were quarantined and why.

In [ ]:
q_manifest_path = QUARANTINE_DIR / "quarantine_manifest.csv"
if q_manifest_path.exists():
    q_df = pd.read_csv(q_manifest_path)
    print(f"Quarantined images: {len(q_df)}\n")
    print("Breakdown by reason:")
    print(q_df["reason"].value_counts().head(10))
    print()
    print("Details (first 20):")
    print(q_df[["class", "split", "reason"]].head(20).to_string(index=False))
else:
    print("No quarantine manifest found — all images were clean!")

In [ ]:
## 4b. Inspect Duplicate Manifest (Cross-Class Duplicates)
Images that appear in multiple classes are data quality issues and are quarantined separately.

In [ ]:
dup_manifest_path = DUPLICATE_DIR / "duplicate_manifest.csv"
if dup_manifest_path.exists():
    dup_df = pd.read_csv(dup_manifest_path)
    print(f"Cross-class duplicates found: {len(dup_df)}")
    print(f"\nBreakdown by class:")
    print(dup_df["class"].value_counts())
    print(f"\nFirst 10 entries:")
    print(dup_df[["class", "split", "reason"]].head(10).to_string(index=False))
else:
    print("No cross-class duplicates found — good data quality!")

## 5. Before / After Visual Comparison
Show raw vs processed for a handful of images to verify cropping, resizing, and mode conversion.

In [ ]:
n_show = 5
samples = manifest.sample(n=min(n_show, len(manifest)), random_state=42)

fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))

for idx, (_, row) in enumerate(samples.iterrows()):
    # Raw (original)
    raw_img = Image.open(row["filepath_original"])
    raw_w, raw_h = raw_img.size
    axes[0, idx].imshow(np.array(raw_img), cmap="gray" if raw_img.mode == "L" else None)
    axes[0, idx].set_title(
        f"RAW | {row['class']}\n{raw_w}×{raw_h}",
        fontsize=9,
    )
    axes[0, idx].axis("off")

    # Processed
    proc_img = Image.open(row["filepath_processed"])
    proc_w, proc_h = proc_img.size
    axes[1, idx].imshow(np.array(proc_img), cmap="gray")
    axes[1, idx].set_title(
        f"PROCESSED | {row['class']}\n{proc_w}×{proc_h} (Grayscale)",
        fontsize=9,
    )
    axes[1, idx].axis("off")

axes[0, 0].set_ylabel("Raw", fontsize=12)
axes[1, 0].set_ylabel("Processed", fontsize=12)
plt.suptitle("Before / After Processing", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()